# Colab: train a supported translation pair
Run cells in order. Select a GPU runtime if available; no runtime duration or GPU availability is guaranteed. Default: English to Twi using [Helsinki-NLP/opus-mt-en-tw](https://huggingface.co/Helsinki-NLP/opus-mt-en-tw). This is not a Twi/French model.
Commit and push this working tree to your GitHub first, or upload the complete updated project to `/content/TW-FR-MT`. Uploading only this notebook does not upload package changes. The original GitHub repository will not contain local changes until pushed. Never embed credentials in this notebook.

In [ ]:
from pathlib import Path
import subprocess
import os
from urllib.parse import urlparse

REPO_URL = 'https://github.com/gyasifred/TW-FR-MT.git'  # Change to your pushed repository.
BRANCH = ''  # Optional branch; existing checkouts are never switched or pulled.
REPO_DIR = Path('/content/TW-FR-MT')
parsed = urlparse(REPO_URL)
if parsed.scheme != 'https' or parsed.hostname != 'github.com' or parsed.username or parsed.password or parsed.query or parsed.fragment:
    raise ValueError('Use a credential-free HTTPS GitHub repository URL.')
if not REPO_DIR.exists():
    command = ['git', 'clone']
    if BRANCH:
        command += ['--branch', BRANCH, '--single-branch']
    subprocess.run(command + ['--', REPO_URL, str(REPO_DIR)], check=True)
else:
    print('Using existing checkout/upload without overwriting it:', REPO_DIR)
if (REPO_DIR / '.git').exists():
    if BRANCH:
        actual = subprocess.check_output(['git', '-C', str(REPO_DIR), 'branch', '--show-current'], text=True).strip()
        if actual != BRANCH:
            raise RuntimeError(f'Existing branch {actual!r} differs from {BRANCH!r}; resolve manually.')
    subprocess.run(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], check=True)
if not (REPO_DIR / 'pyproject.toml').is_file():
    raise RuntimeError('Updated package missing. Push the working tree first or upload the complete updated project.')
os.chdir(REPO_DIR)


In [ ]:
%pip install -e '.[train,app]'
# The project dependency pin is transformers==4.57.1.
from importlib.metadata import version
assert version('transformers') == '4.57.1', 'Check pyproject.toml and restart the runtime after installing.'


## Authorize Drive
First run `00_drive_inventory.ipynb` to inspect authorized metadata and choose real input filenames. Training data: [139FuEvqxcZzVglh0qNBIIDaXusjpgFuh](https://drive.google.com/drive/folders/139FuEvqxcZzVglh0qNBIIDaXusjpgFuh?usp=drive_link). Model outputs: [13irIvPsqnryP_NJ5y6PneFKrQDJs5Qm6](https://drive.google.com/drive/folders/13irIvPsqnryP_NJ5y6PneFKrQDJs5Qm6?usp=drive_link). Open each using your authorized Google account, add shortcuts to My Drive if needed, and choose actual mounted paths below. Folder IDs do not reveal names or filesystem paths. Mounting grants notebook code access to Drive. No public download helper is used; folder contents have not been inspected in this development session.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
DATA_FOLDER_ID = '139FuEvqxcZzVglh0qNBIIDaXusjpgFuh'
MODEL_FOLDER_ID = '13irIvPsqnryP_NJ5y6PneFKrQDJs5Qm6'
DATA_ROOT = Path('/content/drive/MyDrive/CHOOSE_DATA_SHORTCUT_PATH')
DRIVE_ROOT = Path('/content/drive/MyDrive/TW-FR-MT')  # Choose the actual model/output folder path.
SOURCE_FILENAME = 'source_en.txt'  # EXAMPLE ONLY: replace with a name from the authenticated inventory.
TARGET_FILENAME = 'target_tw.txt'  # EXAMPLE ONLY: replace with the aligned Twi file's actual name.
DATA_SELECTION_CONFIRMED = False  # True only after confirming paths, filenames, alignment and languages.
if not DATA_SELECTION_CONFIRMED:
    raise ValueError('Inspect notebook 00, select actual Drive paths/filenames, then confirm the selection.')
for folder in (DATA_ROOT, DRIVE_ROOT):
    if not folder.is_dir():
        raise FileNotFoundError(f'Choose the actual mounted folder/shortcut path: {folder}')
BASE_MODEL = 'Helsinki-NLP/opus-mt-en-tw'
SOURCE_LANG, TARGET_LANG = 'en', 'tw'
SOURCE_FILE = DATA_ROOT / SOURCE_FILENAME
TARGET_FILE = DATA_ROOT / TARGET_FILENAME
assert SOURCE_FILE.is_file(), f'Choose an actual source filename: {SOURCE_FILE}'
assert TARGET_FILE.is_file(), f'Choose an actual target filename: {TARGET_FILE}'
RUN_NAME = 'en-tw-run-001'  # Choose a new name for each fresh experiment.
if Path(RUN_NAME).name != RUN_NAME or RUN_NAME in ('', '.', '..'):
    raise ValueError('RUN_NAME must be one directory name.')
RUN_DIR = DRIVE_ROOT / 'runs' / RUN_NAME
SPLIT_DIR = Path('/content/twfr-splits') / RUN_NAME
LOCAL_DATA_DIR = Path('/content/twfr-inputs') / RUN_NAME
RESUME_FROM_CHECKPOINT = None  # e.g. str(RUN_DIR / 'checkpoint-500'); never just True.
if BASE_MODEL == 'Helsinki-NLP/opus-mt-en-tw' and (SOURCE_LANG, TARGET_LANG) != ('en', 'tw'):
    raise ValueError('This base model supports en -> tw only. Select a compatible model for another pair.')
if RESUME_FROM_CHECKPOINT is None and RUN_DIR.exists() and any(RUN_DIR.iterdir()):
    raise FileExistsError('Run already contains files. Choose a new RUN_NAME or an explicit resume checkpoint.')
if RESUME_FROM_CHECKPOINT is not None and not Path(RESUME_FROM_CHECKPOINT).is_dir():
    raise FileNotFoundError(RESUME_FROM_CHECKPOINT)


## Rebuild clean splits
Primary input is your selected, aligned Drive pair. Copy it to local Colab storage before splitting for faster I/O; the originals remain unchanged. Strict parsing rejects blank rows or unequal lengths instead of silently repairing data. No claim is made about the private Drive corpus's contents.
Optional secondary bundled example: explicitly set `SOURCE_FILE = REPO_DIR / 'TW_FR_EN_corpus/data/total/total_en.txt'` and `TARGET_FILE = REPO_DIR / 'TW_FR_EN_corpus/data/total/total_tw.txt'` (10,708 lines each). Never use archived splits: shipped `test_fr.txt` has 1,070 lines versus 1,071 for Twi/English, with legacy source-key overlaps across train/validation (45), train/test (40), validation/test (2). Never repair or reuse those test files silently. `prepare_splits` groups normalized source texts and deduplicates cleaned pairs. Originals stay unchanged. Only train and validation enter training; test is reserved for final frozen evaluation.
Changing a model or labels does not establish language support. For another pair, verify the model card, aligned data, tokenizer language codes and any required prefix. Do not invent Twi/French tokenizer tokens.

In [ ]:
import shutil
from twfr_mt.data import prepare_splits, read_parallel, validate_splits

# Refuse accidental overwrite; these local copies contain corpus text.
if SPLIT_DIR.exists() or LOCAL_DATA_DIR.exists():
    raise FileExistsError('Inspect existing local inputs/splits before rerunning; use a fresh runtime for resume.')
LOCAL_DATA_DIR.mkdir(parents=True)
local_source = LOCAL_DATA_DIR / 'source.txt'
local_target = LOCAL_DATA_DIR / 'target.txt'
shutil.copy2(SOURCE_FILE, local_source)
shutil.copy2(TARGET_FILE, local_target)
split_result = prepare_splits(local_source, local_target, SPLIT_DIR, seed=42, validation_fraction=0.1, test_fraction=0.1)
assert (SPLIT_DIR / 'manifest.json').is_file()
splits = {name: read_parallel(SPLIT_DIR / f'{name}.source.txt', SPLIT_DIR / f'{name}.target.txt')
          for name in ('train', 'validation', 'test')}
audit = validate_splits(splits)
print('Split counts:', audit['counts'])
print('Removed exact duplicate rows:', split_result['duplicates_removed'])
# Preserve prepared inputs/manifest outside output_dir so the fresh-run guard stays intact.
saved_splits = DRIVE_ROOT / 'prepared_data' / RUN_NAME
if saved_splits.exists():
    local_files = {p.name: p.read_bytes() for p in SPLIT_DIR.iterdir() if p.is_file()}
    saved_files = {p.name: p.read_bytes() for p in saved_splits.iterdir() if p.is_file()}
    if local_files != saved_files:
        raise ValueError('Saved split provenance differs. Do not resume with changed data.')
else:
    shutil.copytree(SPLIT_DIR, saved_splits)
print('Manifest and prepared splits preserved at:', saved_splits)


## Configure training
The next cell uses the package's training configuration. Check paths and settings before running. Checkpoints and the final export go to this Drive run directory; reading split files locally reduces Drive I/O. Drive checkpoint writes can still be slow or interrupted. Resume requires the same model, data split, and training settings as the saved run.

In [ ]:
from twfr_mt.training import TrainingConfig, run_training
# Model revision can be pinned to a Hub commit for reproducibility.
MODEL_REVISION = 'main'
EPOCHS = 3
BATCH_SIZE = 8
GRADIENT_ACCUMULATION_STEPS = 2
LEARNING_RATE = 2e-5
SOURCE_TOKENIZER_LANG = None  # Marian default requires no invented language tokens.
TARGET_TOKENIZER_LANG = None
SOURCE_PREFIX = ''


In [ ]:
config = TrainingConfig(
    base_model=BASE_MODEL, revision=MODEL_REVISION,
    source_lang=SOURCE_LANG, target_lang=TARGET_LANG,
    train_source=str(SPLIT_DIR / 'train.source.txt'),
    train_target=str(SPLIT_DIR / 'train.target.txt'),
    validation_source=str(SPLIT_DIR / 'validation.source.txt'),
    validation_target=str(SPLIT_DIR / 'validation.target.txt'),
    output_dir=str(RUN_DIR), resume_from_checkpoint=RESUME_FROM_CHECKPOINT,
    epochs=EPOCHS, batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS, learning_rate=LEARNING_RATE,
    max_source_length=128, max_target_length=128, seed=42,
    source_tokenizer_lang=SOURCE_TOKENIZER_LANG, target_tokenizer_lang=TARGET_TOKENIZER_LANG,
    source_prefix=SOURCE_PREFIX, save_total_limit=2, device='auto',
)
print(config)


In [ ]:
# This starts training. Do not use test scores for hyperparameter or checkpoint selection.
training_result = run_training(config)
print(training_result)
print('Inspect saved artifacts under:', RUN_DIR)


## After training
Verify a complete model/tokenizer export in `RUN_DIR/final` before opening `02_colab_translate.ipynb`. Point its explicit checkpoint setting at this relative Drive path. Preserve the split manifest and run configuration with the experiment. After freezing the model, generate predictions on the untouched test source and evaluate once; do not feed the test pair into Trainer. No training or Colab execution is claimed by this notebook's authors. See `docs/COLAB.md` and `docs/DEPLOYMENT.md` for persistence and hosting limits.